# V25 upstream association forensics CUDA telemetry

Frozen full-sequence GPU observability audit of exactly 16 opened V24.3 regressions.

**Research question:** When V24.3 regresses, did the correct association never enter the candidate set, or did it enter and lose?

This notebook collects read-only association telemetry and hardware utilization telemetry. It does not train, tune, score for promotion, select or route between V19 and V24, mutate graphs, design an intervention, generate a submission, or authorize deployment.

## 1. Verify Kaggle GPU runtime

Confirm the accelerator and pin the exact Atabey source before inference.

In [ ]:
from pathlib import Path
from collections import Counter
import gzip
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time

BRANCH = "v25-upstream-association-forensics"
EXPECTED_COMMIT = "34e24aec4f1d53b387da978febda93a3aa863013"
ROOT = Path("/tmp/Atabey")
subprocess.run(["nvidia-smi"], check=True)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/drosadocastro-bit/Atabey.git", str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", EXPECTED_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == EXPECTED_COMMIT
RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = f"{ROOT}:{ROOT / 'src'}:{ROOT / 'scripts'}"
print("Atabey commit verified:", actual_commit)

## 2. Install and import dependencies

Install the P100-compatible CUDA 12.6 stack, official evaluator pins, NVML, and plotting dependencies.

In [ ]:
pinned_official = [
    "git+https://github.com/royerlab/tracksdata.git@39dccf3a243e44274759468cb31b2ad9e7fc1d09",
    "git+https://github.com/royerlab/kaggle-cell-tracking-competition.git@075fc5f5a52d11077f9dc2b074644618f26939e2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--index-url", "https://download.pytorch.org/whl/cu126", "torch==2.10.0+cu126", "torchvision==0.25.0+cu126"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pinned_official], check=True)
runtime_packages = [
    "bidict>=0.23.1", "blosc2", "dask", "geff>=1.1.3.1.1", "ilpy>=0.5.1",
    "imagecodecs", "numba", "numcodecs>=0.13", "numpy==2.2.6", "scipy==1.16.3",
    "polars>=1.36.0", "psygnal>=0.14.0", "pyarrow", "rich", "rustworkx>=0.17.1",
    "scikit-image>=0.24.0", "sqlalchemy>=2", "tqdm", "typing-extensions", "zarr>=3.0.10",
    "nvidia-ml-py", "pandas>=2.2", "matplotlib", "seaborn",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *runtime_packages], check=True)
probe_code = '''
import json, numpy as np, scipy, torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
capability = torch.cuda.get_device_capability(0)
required_arch = f"sm_{capability[0]}{capability[1]}"
assert required_arch in torch.cuda.get_arch_list()
assert torch.ones(1, device="cuda").relu().cpu().item() == 1.0
print(json.dumps({"numpy": np.__version__, "scipy": scipy.__version__, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0), "required_arch": required_arch}, sort_keys=True))
'''
runtime_probe = subprocess.run([sys.executable, "-c", probe_code], check=True, capture_output=True, text=True, env=RUN_ENV)
print(runtime_probe.stdout.strip())

import matplotlib.pyplot as plt
import pandas as pd
import pynvml
import seaborn as sns
import torch

## 3. Inspect CUDA device configuration

Record device identity and discover all frozen inputs by exact structure and SHA-256.

In [ ]:
assert torch.cuda.is_available()
pynvml.nvmlInit()
try:
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    device_info = {"device_count": torch.cuda.device_count(), "gpu_model": torch.cuda.get_device_name(0), "cuda_runtime": torch.version.cuda, "compute_capability": torch.cuda.get_device_capability(0), "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory, "driver": pynvml.nvmlSystemGetDriverVersion()}
finally:
    pynvml.nvmlShutdown()
print(json.dumps(device_info, indent=2, sort_keys=True))

INPUT_ROOT = Path("/kaggle/input")
EXPECTED_CHECKPOINT_SHA256 = "02e1d65756c3dc5928f68a66a8b0ef99be2a6905fa7bc017aa1d87dbe632fd03"
EXPECTED_PREDICTOR_SHA256 = "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9"
train_candidates = []
for pattern in ("*/train", "*/*/train", "*/*/*/train"):
    for candidate in INPUT_ROOT.glob(pattern):
        if candidate.is_dir() and len(list(candidate.glob("*.zarr"))) == 199 and len(list(candidate.glob("*.geff"))) == 199:
            train_candidates.append(candidate)
train_candidates = sorted(set(train_candidates))
assert len(train_candidates) == 1, train_candidates
TRAIN_DIR = train_candidates[0]
auxiliary_roots = sorted(root for root in INPUT_ROOT.iterdir() if root.is_dir() and not TRAIN_DIR.is_relative_to(root))
predictors = [path for root in auxiliary_roots for path in root.rglob("predict_unet_transformer.py") if path.parent.name == "scripts" and hashlib.sha256(path.read_bytes()).hexdigest() == EXPECTED_PREDICTOR_SHA256]
assert len(predictors) == 1, predictors
SUPPORT_REPO = predictors[0].parents[1]
weights = [path for root in auxiliary_roots for path in root.rglob("edge_predictor_best.pth") if (path.parent / "config.json").exists() and hashlib.sha256(path.read_bytes()).hexdigest() == EXPECTED_CHECKPOINT_SHA256]
assert len(weights) == 1, weights
WEIGHTS = weights[0]
checkpoint_config = json.loads((WEIGHTS.parent / "config.json").read_text(encoding="utf-8"))
assert checkpoint_config["window_size"] == 2
assert checkpoint_config["downsample"] == [1, 4, 4]
assert checkpoint_config["unet_out_channels"] == 32
assert checkpoint_config["pool_kernel_um"] == 5.0
print(TRAIN_DIR, SUPPORT_REPO, WEIGHTS, sep="\n")

## 4. Collect NVIDIA-SMI telemetry

Capture a pre-run snapshot of utilization, temperature, power, clocks, and memory.

In [ ]:
smi_fields = ["timestamp", "name", "utilization.gpu", "utilization.memory", "temperature.gpu", "power.draw", "clocks.sm", "clocks.mem", "memory.used", "memory.total"]
smi_snapshot = subprocess.check_output(["nvidia-smi", f"--query-gpu={','.join(smi_fields)}", "--format=csv,noheader,nounits"], text=True).strip()
assert smi_snapshot
print(smi_snapshot)

## 5. Build a Python telemetry sampler

Validate the frozen contract and define a one-second background NVML sampler.

In [ ]:
focused_tests = [ROOT / "tests/test_association_forensics.py", ROOT / "tests/test_official_association_forensics.py", ROOT / "tests/test_v25_upstream_association_forensics_contract.py", ROOT / "tests/test_v25_upstream_association_forensics_runner.py"]
subprocess.run([sys.executable, "-m", "pytest", "-q", *map(str, focused_tests)], check=True, env=RUN_ENV)
AUTHORIZE_V25_OBSERVABILITY = True
assert AUTHORIZE_V25_OBSERVABILITY is True
CONTRACT_PATH = ROOT / "tests/fixtures/v25_upstream_association_forensics.json"
PREREGISTRATION_PATH = ROOT / "V25_UPSTREAM_ASSOCIATION_FORENSICS_PREREGISTRATION.md"
TAXONOMY_PATH = ROOT / "V25_FAILURE_TAXONOMY_AUDIT.md"
contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
sample_ids = contract["cohort"]["sample_ids"]
assert contract["status"] == "preregistered_observability_only_not_executed"
assert len(sample_ids) == len(set(sample_ids)) == contract["cohort"]["sample_count"] == 16
for boundary in ("score_claim", "promotion_claim", "selector_claim", "automatic_v19_v24_routing", "threshold_or_penalty_tuning", "graph_mutation", "submission_authorized"):
    assert contract["boundaries"][boundary] is False, boundary
OUTPUT_DIR = Path("/kaggle/working/v25_upstream_forensics")

def sample_gpu(stop_event, records, interval_seconds=1.0):
    pynvml.nvmlInit()
    started = time.monotonic()
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        while not stop_event.is_set():
            utilization = pynvml.nvmlDeviceGetUtilizationRates(handle)
            memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
            records.append({"elapsed_seconds": time.monotonic() - started, "timestamp_unix_seconds": time.time(), "gpu_utilization_percent": utilization.gpu, "memory_utilization_percent": utilization.memory, "memory_used_mib": memory.used / 1048576, "memory_total_mib": memory.total / 1048576, "temperature_c": pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU), "power_draw_w": pynvml.nvmlDeviceGetPowerUsage(handle) / 1000, "sm_clock_mhz": pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_SM), "memory_clock_mhz": pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_MEM)})
            stop_event.wait(interval_seconds)
    finally:
        pynvml.nvmlShutdown()

## 6. Run the CUDA workload

Run a seeded matrix probe and the unchanged 16-sample, full-sequence V25 observer under hardware sampling.

In [ ]:
command = [sys.executable, "-u", str(ROOT / "scripts/run_v25_upstream_association_forensics.py"), "--train-dir", str(TRAIN_DIR), "--support-repo", str(SUPPORT_REPO), "--weights", str(WEIGHTS), "--contract", str(CONTRACT_PATH), "--output-dir", str(OUTPUT_DIR), "--unet-batch-size", "4", "--resume"]
telemetry_records = []
telemetry_stop = threading.Event()
telemetry_thread = threading.Thread(target=sample_gpu, args=(telemetry_stop, telemetry_records), daemon=True)
telemetry_thread.start()
started = time.time()
try:
    torch.manual_seed(25)
    left = torch.randn((2048, 2048), device="cuda")
    right = torch.randn((2048, 2048), device="cuda")
    probe_checksum = float((left @ right).relu().mean().cpu())
    torch.cuda.synchronize()
    del left, right
    torch.cuda.empty_cache()
    print("CUDA matrix probe:", probe_checksum)
    subprocess.run(command, check=True, env=RUN_ENV)
finally:
    elapsed_seconds = time.time() - started
    telemetry_stop.set()
    telemetry_thread.join(timeout=10)
    assert not telemetry_thread.is_alive()
print("Elapsed hours:", elapsed_seconds / 3600)

## 7. Visualize GPU metrics

Validate all association artifacts and plot utilization, allocated memory, temperature, power, and clocks.

In [ ]:
summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
provenance = json.loads((OUTPUT_DIR / "provenance.json").read_text(encoding="utf-8"))
assert summary["status"] == "V25_OBSERVABILITY_COMPLETE"
assert summary["completed_samples"] == 16
assert summary["score_claim"] is summary["selector_claim"] is summary["graph_mutation"] is False
assert {row["sample_id"] for row in summary["samples"]} == set(sample_ids)
assert provenance["score_claim"] is provenance["graph_mutation"] is False
assert provenance["max_timepoints"] is None and provenance["unet_batch_size"] == 4
sample_paths = sorted((OUTPUT_DIR / "samples").glob("*.json.gz"))
assert len(sample_paths) == 16
failure_counts = Counter()
for sample_path in sample_paths:
    with gzip.open(sample_path, "rt", encoding="utf-8") as handle:
        record = json.load(handle)
    assert record["sample_id"] == sample_path.name.removesuffix(".json.gz")
    assert record["graph_mutated"] is record["score_claim"] is False
    failure_counts.update(record["failure_class_counts"])
telemetry_frame = pd.DataFrame.from_records(telemetry_records)
required = ["elapsed_seconds", "gpu_utilization_percent", "memory_used_mib", "temperature_c", "power_draw_w", "sm_clock_mhz", "memory_clock_mhz"]
assert not telemetry_frame.empty and telemetry_frame[required].notna().all().all()
assert telemetry_frame["elapsed_seconds"].is_monotonic_increasing
print(json.dumps(dict(failure_counts), indent=2, sort_keys=True))
sns.set_theme(style="whitegrid")
plot_specs = [("gpu_utilization_percent", "GPU utilization (%)"), ("memory_used_mib", "Allocated memory (MiB)"), ("temperature_c", "Temperature (C)"), ("power_draw_w", "Power (W)"), ("sm_clock_mhz", "SM clock (MHz)"), ("memory_clock_mhz", "Memory clock (MHz)")]
figure, axes = plt.subplots(3, 2, figsize=(14, 11), sharex=True)
for axis, (column, label) in zip(axes.flat, plot_specs):
    sns.lineplot(data=telemetry_frame, x="elapsed_seconds", y=column, ax=axis, errorbar=None)
    axis.set_ylabel(label)
figure.suptitle("V25 CUDA hardware telemetry")
figure.tight_layout()
TELEMETRY_PLOT = Path("/kaggle/working/v25_cuda_telemetry.png")
figure.savefig(TELEMETRY_PLOT, dpi=150, bbox_inches="tight")
plt.show()

## 8. Export telemetry data

Save validated CSV/JSON hardware traces and bundle them with deterministic association records. Hardware telemetry is operational evidence and varies by device load.

In [ ]:
TELEMETRY_CSV = Path("/kaggle/working/v25_cuda_telemetry.csv")
TELEMETRY_JSON = Path("/kaggle/working/v25_cuda_telemetry.json")
telemetry_frame.to_csv(TELEMETRY_CSV, index=False)
TELEMETRY_JSON.write_text(json.dumps(telemetry_frame.to_dict(orient="records"), indent=2, sort_keys=True) + "\n", encoding="utf-8")
assert TELEMETRY_CSV.stat().st_size > 0 and TELEMETRY_JSON.stat().st_size > 0
BUNDLE = Path("/kaggle/working/v25_upstream_forensics_outputs")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR, BUNDLE / "run")
for path in (CONTRACT_PATH, PREREGISTRATION_PATH, TAXONOMY_PATH, TELEMETRY_CSV, TELEMETRY_JSON, TELEMETRY_PLOT):
    shutil.copy2(path, BUNDLE / path.name)
run_record = {"mode": "v25_upstream_association_forensics", "atabey_commit": EXPECTED_COMMIT, "checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256, "predictor_sha256": EXPECTED_PREDICTOR_SHA256, "sample_count": 16, "elapsed_seconds": elapsed_seconds, "status": summary["status"], "gpu_device": device_info, "hardware_telemetry_sample_count": len(telemetry_frame), "hardware_telemetry_is_deterministic": False, "association_artifacts_use_deterministic_serialization": True, "no_training": True, "score_claim": False, "selector_enabled": False, "graph_mutation": False, "submission_authorized": False}
(BUNDLE / "notebook_run_record.json").write_text(json.dumps(run_record, indent=2, sort_keys=True) + "\n", encoding="utf-8")
archive = shutil.make_archive(str(BUNDLE), "zip", BUNDLE)
print("Download:", archive)
print("Interpretation boundary: preserve unresolved cases; no tuning, automatic V19/V24 selection, graph mutation, intervention, or submission is authorized.")